# 01 — Setup and Connect to Azure AI Foundry

This notebook guides you through connecting to an Azure AI Foundry project
using the **Microsoft Agent Framework**. It verifies your credentials, endpoint,
and model deployment are working correctly.

## Configuration

Set your agent name below. Change this to work with any registered agent.

In [1]:
import sys, pathlib

# Ensure the local project root is first on sys.path so our `agents`
# package takes priority over the openai-agents site-package.
_project_root = str(pathlib.Path.cwd().parent)
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

# Change this to any registered agent name
AGENT_NAME = "code-helper"

## Install Dependencies

In [2]:
!uv sync

Resolved 226 packages in 1ms
Checked 221 packages in 1ms


## Load Environment

Loads `AZURE_AI_PROJECT_ENDPOINT` from your `.env` file (in the repo root).
Make sure you've copied `.env.example` to `.env` and filled in your endpoint.

---

In [3]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")
AZURE_AI_PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"]
print(f"✓ Endpoint loaded: {AZURE_AI_PROJECT_ENDPOINT[:50]}…")

✓ Endpoint loaded: https://ai-openai-arizona.openai.azure.us/…


## Verify Azure Credentials

Tests that `DefaultAzureCredential` can authenticate.
Run `az login` first if you haven't already.

In [4]:
from azure.identity import DefaultAzureCredential
import os

# Use the correct authority and scope for sovereign clouds
authority = os.environ.get("AZURE_AUTHORITY_HOST")
cred_kwargs = {"authority": authority} if authority else {}
credential = DefaultAzureCredential(**cred_kwargs)

# Azure Government uses .azure.us scopes
scope = (
    "https://cognitiveservices.azure.us/.default"
    if authority and "microsoftonline.us" in authority
    else "https://cognitiveservices.azure.com/.default"
)
token = credential.get_token(scope)
print(f"✓ Credential authenticated successfully (scope: {scope})")
print(f"  Token expires: {token.expires_on}")

✓ Credential authenticated successfully (scope: https://cognitiveservices.azure.us/.default)
  Token expires: 1775849724


## Create Chat Client

Creates an `AzureOpenAIResponsesClient` — the same client the agent factory uses.
This verifies the endpoint and model deployment are reachable.

In [5]:
from agents.registry import REGISTRY
from agents._base.client import get_chat_client

# Look up the agent's config to get the deployment name
entry = REGISTRY.get_agent(AGENT_NAME)
config = entry.config_class()

print(f"Agent:      {config.agent_name}")
print(f"Deployment: {config.agent_deployment_name}")
print(f"Endpoint:   {config.azure_ai_project_endpoint}")

client = get_chat_client(
    endpoint=config.azure_ai_project_endpoint,
    deployment_name=config.agent_deployment_name,
)
print(f"\n✓ Chat client created: {type(client).__name__}")

Agent:      code-helper
Deployment: gpt-5.1-2
Endpoint:   https://ai-openai-arizona.openai.azure.us/

✓ Chat client created: AzureOpenAIResponsesClient


## Assemble a Test Agent

Uses the agent factory to create an agent in-process — the same path
as `run_agent.py` and `app.py`.

In [6]:
from agents._base.agent_factory import create_agent

agent = create_agent(config)
print(f"✓ Agent assembled: {config.agent_name}")
print(f"  Type: {type(agent).__name__}")

✓ Agent assembled: code-helper
  Type: Agent


## Quick Smoke Test

Send a simple prompt to verify the full pipeline works end-to-end:
environment → credentials → client → agent → model → response.

In [7]:
result = await agent.run("Say 'Connection OK' and nothing else.")
response_text = result.text if hasattr(result, "text") else str(result)
print(f"✓ Agent response: {response_text}")

/home/kiran/projects/agentframework-automation/.venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ai-search-arizona.search.azure.us'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


✓ Agent response: Connection OK


## List All Registered Agents

Shows every agent in the registry — these are all available for use.

In [8]:
print("Registered agents:")
for entry in REGISTRY.list_agents():
    marker = " ← current" if entry.name == AGENT_NAME else ""
    print(f"  • {entry.name} ({entry.config_class.__name__}){marker}")

Registered agents:
  • code-helper (CodeHelperConfig) ← current
  • doc-assistant (DocAssistantConfig)


## Endpoint Diagnostics

Parse the endpoint URL to show project details for troubleshooting.

In [9]:
endpoint = config.azure_ai_project_endpoint.rstrip("/")
parts = endpoint.split("/api/projects/")
if len(parts) == 2:
    hub_url = parts[0]
    project_name = parts[1]
    print(f"Hub endpoint:  {hub_url}")
    print(f"Project name:  {project_name}")
    print(f"\n→ In Azure AI Foundry portal, verify project '{project_name}'")
    print(f"  has a deployment named '{config.agent_deployment_name}'")
else:
    print(f"Endpoint: {endpoint}")

Endpoint: https://ai-openai-arizona.openai.azure.us


## Next Steps

- Continue to **02_build_and_run_agent.ipynb** to have a full conversation with an agent
- See **03_scaffold_agent.ipynb** to create a brand new agent